In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import create_retriever_tool
from langchain.agents import create_agent
from langchain_chroma import Chroma
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

True

In [2]:
hdfc_loader = PyPDFLoader("hdfcbank.pdf")
hdfc_pages = hdfc_loader.load_and_split()
print(f"{len(hdfc_pages)} pages loaded from the PDF.")

21 pages loaded from the PDF.


In [3]:
hdfc_pages[13]

Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260718142515', 'source': 'hdfcbank.pdf', 'total_pages': 16, 'page': 10, 'page_label': '11'}, page_content='HDFC BANK\nWe understand your world\n7 During the quarter ended June 30, 2026, the Bank has allotted 79,59,164 equity shares pursuant to the exercise of\noptions / units under the approved employee stock option schemes / employee stock incentive master scheme.\n8 During the quarter ended June 30, 2025 and year ended March 31, 2026, the Bank made a floating provision of\n9,000.00 crore in line with the Board approved policy.\n9 On June 25, 2025, the Bank\'s subsidiary company, HDB Financial Services Limited ("HDBFS") launched its initial public\noffering ("IPO"), comprised of a fresh issuance of equity shares aggregating to 2,500.00 crore and an offer for sale\n("OFS") of equity shares by the Bank, aggregating to₹ 10,000.00 crore. Under the OFS, the Bank divested\n13,51,35,135 equity shares of 10/- e

In [4]:
hdfc_store = Chroma.from_documents(hdfc_pages, collection_name="financial_report", persist_directory="./chroma")
retriever = hdfc_store.as_retriever(search_type="similarity", 
                                       search_kwargs={"k" : 5})

In [5]:
hdfc_tool = create_retriever_tool(
  retriever,
  "hdfc_search",
  "Search for information about HDFC BANK financial statements."
  "Use this for any questions about HDFC revenue, income, "
  "expenses, or other financial metrics."
)

In [6]:
sbi_loader = PyPDFLoader("sbi.pdf")
sbi_pages = sbi_loader.load_and_split()
print(f"{len(sbi_pages)} pages loaded from the PDF.")

2 pages loaded from the PDF.


In [7]:
sbi_store = Chroma.from_documents(sbi_pages, collection_name="financial_report", persist_directory="./chroma")
sbi_retriever = sbi_store.as_retriever(search_type="similarity", 
                                       search_kwargs={"k" : 5})

In [8]:
sbi_tool = create_retriever_tool(sbi_retriever,
                                 "sbi_search",
  "Search for information about SBI BANK financial statements."
  "Use this for any questions about SBI revenue, income, "
  "expenses, or other financial metrics.")

In [9]:
llm = ChatGoogleGenerativeAI(
  model = "gemini-2.5-flash",
  temperature = 0.1
)

In [10]:
agent = create_agent(llm, [hdfc_tool, sbi_tool])

In [11]:
query = "How was HDFC's total revenue,which quarter had most returns and how does it compare to previous year?"

result = agent.invoke({"messages" : [("user", query)]})
resp = result["messages"][-1].content
display(Markdown(resp[0]['text'].replace('\\n', '\n')))

HDFC's total revenue (Total Income) for the quarter ended June 30, 2026, was 133,110.36 crore.

The quarter with the most returns (Net profit for the period) was March 31, 2026, with a net profit of 20,350.76 crore.

Comparing the total revenue for the quarter ended June 30, 2026 (133,110.36 crore) to the same quarter in the previous year (June 30, 2025, which was 133,054.97 crore), there was a slight increase in revenue.

In [13]:
compare_query = "Compare the financial results of SBI bank and HDFC bank, whose returns are more in which quarter and whats the net profit?"

result = agent.invoke({"messages" : [("user", compare_query)]})
resp = result["messages"][-1].content
display(Markdown(resp[0]['text'].replace('\\n', '\n')))

I am sorry, but I cannot compare the financial results of SBI Bank and HDFC Bank at this moment. The search results for both banks returned identical information, indicating a potential issue with the data source. Both results state:

*   **Net Profit for Q4FY26:** ₹19,684 crores
*   **Return on Assets (ROA) for FY26:** 1.12%
*   **Return on Equity (ROE) for FY26:** 18.57%

Since the data is identical for both, I cannot provide a meaningful comparison.